In [1]:
import geopandas as gpd
import pandas as pd

In [2]:
buildings = gpd.read_parquet('output/buildings_rovaniemi.parquet')

In [3]:
print(buildings.shape)
print(buildings.crs)
print(buildings.head(2))

(18045, 3)
{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "scope": 

In [4]:
depot = gpd.read_file('output/depot_rovaniemi.gpkg')

In [5]:
print(depot.shape)
print(depot.crs)
print(depot.head(2))

(1, 9)
EPSG:4326
   postal_code  placename  longitude   latitude  Diesel  Electric  Gas  Total  \
0        96320  Rovaniemi  25.620261  66.479989    28.0       5.0  4.0   37.0   

                    geometry  
0  POINT (25.62026 66.47999)  


In [6]:
charging_stations = gpd.read_file('output/ev_stations_rovaniemi.gpkg')           



In [7]:
print(charging_stations.shape)
print(charging_stations.crs)
print(charging_stations.head(2))

(1, 9)
EPSG:4326
   postal_code  placename  longitude   latitude  Diesel  Electric  Gas  Total  \
0        96320  Rovaniemi  25.620261  66.479989    28.0       5.0  4.0   37.0   

                    geometry  
0  POINT (25.62026 66.47999)  


In [8]:
from sklearn.cluster import KMeans
import geopandas as gpd
import numpy as np

# How many clusters — aim for <100 for ORS free tier
N_CLUSTERS = 56

coords = np.column_stack([buildings['longitude'], buildings['latitude']])

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init='auto')
buildings['cluster'] = kmeans.fit_predict(coords)

# Get centroid of each cluster as the delivery stop
cluster_centroids = (
    buildings.groupby('cluster')[['longitude', 'latitude']]
    .mean()
    .reset_index()
)

print(cluster_centroids.shape)
print(cluster_centroids.head())

(56, 3)
   cluster  longitude   latitude
0        0  25.064817  66.422819
1        1  25.780868  66.521293
2        2  26.658924  66.618168
3        3  25.189049  66.863749
4        4  25.563163  66.452468


In [9]:
# Step 2: Build coordinate list
# Order matters: depot first, then delivery stops, then charging stations

depot_coords = [[depot['longitude'].values[0], depot['latitude'].values[0]]]

delivery_coords = cluster_centroids[['longitude', 'latitude']].values.tolist()
delivery_coords = [[lon, lat] for lon, lat in delivery_coords]

charging_coords = [[charging_stations['longitude'].values[0], charging_stations['latitude'].values[0]]]

# Combine in order
all_coords = depot_coords + delivery_coords + charging_coords

# Track index ranges for each type
depot_index = 0
delivery_indices = list(range(1, 1 + len(delivery_coords)))
charging_indices = list(range(1 + len(delivery_coords), 1 + len(delivery_coords) + len(charging_coords)))

print(f"Total locations: {len(all_coords)}")
print(f"Depot index: {depot_index}")
print(f"Delivery stops: {len(delivery_indices)} (indices {delivery_indices[0]}–{delivery_indices[-1]})")
print(f"Charging stations: {len(charging_indices)} (indices {charging_indices})")

Total locations: 58
Depot index: 0
Delivery stops: 56 (indices 1–56)
Charging stations: 1 (indices [57])


In [10]:
import requests
import numpy as np

ORS_API_KEY = "5b3ce3597851110001cf624835fda19e08524774bdb86e5d1c5e64a5"

def get_distance_matrix(coordinates, api_key):
    url = "https://api.openrouteservice.org/v2/matrix/driving-car"
    headers = {
        "Authorization": api_key,
        "Content-Type": "application/json"
    }
    body = {
        "locations": coordinates,
        "metrics": ["distance"],
        "units": "m"
    }
    r = requests.post(url, json=body, headers=headers)
    
    if r.status_code != 200:
        print(f"Error {r.status_code}: {r.text}")
        return None
    
    return np.array(r.json()["distances"])

distance_matrix = get_distance_matrix(all_coords, ORS_API_KEY)

if distance_matrix is not None:
    print(f"Matrix shape: {distance_matrix.shape}")
    print(f"Example — depot to first delivery stop: {distance_matrix[0][1]/1000:.2f} km")
    print(f"Example — depot to charging station: {distance_matrix[0][charging_indices[0]]/1000:.2f} km")

Matrix shape: (58, 58)
Example — depot to first delivery stop: 33.67 km
Example — depot to charging station: 0.00 km


In [ ]:
!pip install ortools

Defaulting to user installation because normal site-packages is not writeable


: 

In [ ]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

def create_data_model(distance_matrix, num_vehicles, vehicle_range_m):
    data = {}
    # Convert to integers (OR-Tools requires integers)
    data['distance_matrix'] = (distance_matrix * 1).astype(int).tolist()
    data['num_vehicles'] = num_vehicles
    data['depot'] = 0
    data['vehicle_max_distance'] = vehicle_range_m
    return data

def solve_vrp(data):
    manager = pywrapcp.RoutingIndexManager(
        len(data['distance_matrix']),
        data['num_vehicles'],
        data['depot']
    )
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return data['distance_matrix'][from_node][to_node]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # Range constraint
    routing.AddDimension(
        transit_callback_index,
        0,
        data['vehicle_max_distance'],
        True,
        'Distance'
    )
    distance_dimension = routing.GetDimensionOrDie('Distance')
    distance_dimension.SetGlobalSpanCostCoefficient(100)

    search_params = pywrapcp.DefaultRoutingSearchParameters()
    search_params.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    search_params.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    search_params.time_limit.seconds = 30

    solution = routing.SolveWithParameters(search_params)
    return routing, manager, solution

def print_solution(manager, routing, solution, data):
    total_distance = 0
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        route_distance = 0
        route = []
        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            route.append(node)
            next_index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(index, next_index, vehicle_id)
            index = next_index
        route.append(0)  # return to depot
        print(f"Vehicle {vehicle_id}: {len(route)-2} stops | {route_distance/1000:.1f} km | route: {route}")
        total_distance += route_distance
    print(f"\nTotal distance all vehicles: {total_distance/1000:.1f} km")

# Run with 5 vehicles and 120 km winter range
data = create_data_model(distance_matrix, num_vehicles=5, vehicle_range_m=120000)
routing, manager, solution = solve_vrp(data)

if solution:
    print("✓ Solution found\n")
    print_solution(manager, routing, solution, data)
else:
    print("✗ No solution found — 5 vehicles with 120 km range cannot cover all stops")